## 1 · Imports

In [ ]:
import math
import time

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import torch
import torch.nn as nn
from sklearn.datasets import make_moons
from tqdm.auto import tqdm

from torchcfm.conditional_flow_matching import (
    ExactOptimalTransportConditionalFlowMatcher,
    ConditionalFlowMatcher,
)

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is required for this notebook — no GPU detected.")

device = torch.device("cuda")

torch.manual_seed(42)
torch.cuda.manual_seed_all(42)
np.random.seed(42)

print(f"device: {device}  ({torch.cuda.get_device_name(device)})")

## 2 . Loading JAX Flow Map model and converting to Pytorch

In [ ]:
"""
Load the JAX/Flax two-moons flow-map checkpoint and convert it to a
matching PyTorch model.

Architecture (from configs/two_moons.py + network_utils.py):
  FlowMapMLP
  - Input : [s, t, x/rescale]  →  dim 4  (2 time scalars + 2-D point)
  - phi_mlp (MLP):
      Linear(4  → 256) + GELU
      Linear(256 → 256) + GELU  (× n_hidden = 3)
      Linear(256 → 2)
  - phi = rescale * mlp(inp)
  - X(s,t,x) = x + (t-s) * phi

Key conversion detail:
  Flax Dense kernel shape : [in, out]
  PyTorch Linear weight   : [out, in]  → must transpose
"""

import sys, os
sys.path.insert(0, "/home/shervink/flow_matching/flow-maps/py")

import numpy as np
import torch
import torch.nn as nn
from flax.serialization import msgpack_restore
from common.datasets import sample_two_moons

# ── paths ────────────────────────────────────────────────────────────────────
CKPT = "/home/shervink/flow_matching/flow-maps/py/checkpoints/two_moons_paper_lsd_15.pkl"

# Must match training: adaptive gaussian_scale sets network.rescale = std(target)
RESCALE = float(np.std(sample_two_moons(500_000)))
print(f"RESCALE (from training data std): {RESCALE:.4f}")

# ── 1. load raw parameter dict ────────────────────────────────────────────────
with open(CKPT, "rb") as f:
    raw = msgpack_restore(f.read())

# The EMATrainState serialises as:
#   raw["params"]      → TrainState.params  (the variable collection)
#   raw["ema_params"]  → dict keyed by EMA factor
#
# Use the EMA-smoothed weights (0.999) for inference.
ema_key = list(raw["ema_params"].keys())[0]
print(f"EMA keys available: {list(raw['ema_params'].keys())}")
print(f"Using EMA key: {ema_key!r}   step={int(raw['step'])}")

phi = raw["ema_params"][ema_key]["params"]["flow_map"]["phi_mlp"]
print("phi_mlp Dense layers found:", sorted(phi.keys()))
for k in sorted(phi.keys()):
    print(f"  {k}: kernel {phi[k]['kernel'].shape}  bias {phi[k]['bias'].shape}")

# ── 2. define matching PyTorch model ─────────────────────────────────────────
class FlowMapMLP(nn.Module):
    """
    Replicates FlowMapMLP from network_utils.py.
    forward(s, t, x) returns X(s,t,x) = x + (t-s) * phi(s,t,x).
    s, t : scalars or (N,) tensors
    x    : (N, 2) tensor
    """
    def __init__(self, n_hidden=3, n_neurons=256, input_dim=4, output_dim=2,
                 rescale=0.5):
        super().__init__()
        self.rescale = rescale
        layers = []
        in_dim = input_dim
        # first layer
        layers += [nn.Linear(in_dim, n_neurons), nn.GELU()]
        # n_hidden hidden layers
        for _ in range(n_hidden):
            layers += [nn.Linear(n_neurons, n_neurons), nn.GELU()]
        # output layer
        layers += [nn.Linear(n_neurons, output_dim)]
        self.phi_mlp = nn.Sequential(*layers)

    def calc_b(self, t, x):
        """Induced velocity on the diagonal: b(t, x) = phi(t, t, x)."""
        if t.dim() == 0:
            t = t.expand(x.shape[0])
        st = torch.stack([t, t], dim=-1)
        inp = torch.cat([st, x / self.rescale], dim=-1)
        return self.rescale * self.phi_mlp(inp)

    def forward(self, s, t, x):
        # x : (N, 2);  s, t : () or (N,)
        if s.dim() == 0:
            s = s.expand(x.shape[0])
            t = t.expand(x.shape[0])
        st = torch.stack([s, t], dim=-1)          # (N, 2)
        inp = torch.cat([st, x / self.rescale], dim=-1)   # (N, 4)
        phi = self.rescale * self.phi_mlp(inp)     # (N, 2)
        return x + (t - s).unsqueeze(-1) * phi     # X(s,t,x)


# ── 3. copy Flax weights → PyTorch ───────────────────────────────────────────
def flax_to_pt(flax_dense, pt_linear):
    """Copy a single Flax Dense layer into a PyTorch Linear layer."""
    W = torch.from_numpy(np.array(flax_dense["kernel"]))   # [in, out]
    b = torch.from_numpy(np.array(flax_dense["bias"]))     # [out]
    pt_linear.weight.data.copy_(W.T)   # PyTorch wants [out, in]
    pt_linear.bias.data.copy_(b)

model = FlowMapMLP(n_hidden=3, n_neurons=256, input_dim=4, output_dim=2,
                   rescale=RESCALE).to(device)

# phi_mlp Sequential layout:
#   [0] Linear  [1] GELU  [2] Linear  [3] GELU  ... [2k] Linear  [2k+1] GELU  [last] Linear
# Dense_0 → layer 0, Dense_1 → layer 2, Dense_2 → layer 4, ..., Dense_4 → layer 8
dense_to_seq = {
    "Dense_0": 0,
    "Dense_1": 2,
    "Dense_2": 4,
    "Dense_3": 6,
    "Dense_4": 8,
}
for dense_name, seq_idx in dense_to_seq.items():
    flax_to_pt(phi[dense_name], model.phi_mlp[seq_idx])

print("\nWeights copied successfully.")

# ── 4. quick sanity check ─────────────────────────────────────────────────────
model.eval()
with torch.no_grad():
    x_test = torch.randn(5, 2, device=device)
    s_test = torch.zeros(5, device=device)
    t_test = torch.ones(5, device=device)
    out = model(s_test, t_test, x_test)
    print(f"Test output shape: {out.shape}  (expected (5, 2))")
    print(f"Sample output:\n{out.cpu().numpy().round(4)}")




## 3 . Sampling Test

In [ ]:
"""
Visualise flow-map samples at 1, 5, 10, 25 NFE (+ base/target panel).
Mirrors make_lowd_plot() in common/logging.py.

Generated points are NOT post-normalized — they are raw flow-map outputs.
Any mismatch with the target is usually from (1) wrong RESCALE at inference,
or (2) plotting target data with a different sampler/normalization than training.
"""

import sys
sys.path.insert(0, "/home/shervink/flow_matching/flow-maps/py")

import numpy as np
import matplotlib.pyplot as plt
from common.datasets import sample_two_moons

# ── helpers ───────────────────────────────────────────────────────────────────
def sample_rho0(n, rescale=RESCALE, device=device):
    """Same base distribution as training: N(0, rescale^2 I)."""
    return rescale * torch.randn(n, 2, device=device)

@torch.no_grad()
def flow_map_sample(model, x0, N):
    """Apply the flow map in N equal steps: t_0=0 → t_1 → … → t_N=1."""
    ts = torch.linspace(0.0, 1.0, N + 1, device=x0.device)
    x = x0.clone()
    for i in range(N):
        s = ts[i].expand(x.shape[0])
        t = ts[i + 1].expand(x.shape[0])
        x = model(s, t, x)
    return x

def make_target(n_plot=5_000, n_total=500_000, noise=0.05):
    """Use the exact training sampler, then subsample for plotting."""
    all_samples = sample_two_moons(n_total, noise=noise)
    idx = np.random.choice(n_total, size=n_plot, replace=False)
    return all_samples[idx]

def sample_target(n_plot=5_000, n_total=500_000, noise=0.05):
    """Use the exact training sampler, then subsample for plotting."""
    all_samples = sample_two_moons(n_total, noise=noise)
    idx = np.random.choice(n_total, size=n_plot, replace=False)
    return all_samples[idx]

# ── data ──────────────────────────────────────────────────────────────────────
N_PLOT  = 5_000
steps   = [1, 5, 10, 25]

x1s  = make_target(N_PLOT)                          # target (numpy)
x0s  = sample_rho0(N_PLOT)                          # noise  (torch, GPU)

print(f"RESCALE used: {RESCALE:.4f}")
print(f"target std: {x1s.std():.4f}   base std: {x0s.std().item():.4f}")

xhats = {}
model.eval()
for N in steps:
    xhats[N] = flow_map_sample(model, x0s, N).cpu().numpy()
    print(f"{N:2d}-step generated std: {xhats[N].std():.4f}")

x0s_np = x0s.cpu().numpy()

# ── plot ──────────────────────────────────────────────────────────────────────
titles = ["base & target"] + [f"${N}$-step" for N in steps]
ncols  = len(titles)
fw, fh = 4, 4
fontsize = 12.5

fig, axs = plt.subplots(1, ncols, figsize=(fw * ncols, fh),
                        sharex=True, sharey=True, constrained_layout=True)

for ax in axs:
    ax.set_xlim(-1.5, 1.5)
    ax.set_ylim(-1.5, 1.5)
    ax.set_aspect("equal")
    ax.grid(color="0.90", alpha=0.4)
    ax.tick_params(labelsize=fontsize)

for jj, (ax, title) in enumerate(zip(axs, titles)):
    ax.set_title(title, fontsize=fontsize)
    ax.scatter(x1s[:, 0], x1s[:, 1], s=0.3, alpha=0.4, c="C0", label="target")
    if jj == 0:
        ax.scatter(x0s_np[:, 0], x0s_np[:, 1], s=0.3, alpha=0.4,
                   c="black", label="base")
    else:
        N = steps[jj - 1]
        ax.scatter(xhats[N][:, 0], xhats[N][:, 1], s=0.3, alpha=0.4,
                   c="black", label="generated")

axs[0].legend(markerscale=10, fontsize=10, loc="upper right")
plt.suptitle("FlowMapMLP (LSD, two-moons) — N-step samples", fontsize=13)
plt.show()

## 4 . Sampling Test with induced vector field (X(t,t))

In [ ]:
from typing import Optional, Tuple, Union

@torch.no_grad()
def euler_sample(
    model,
    n: int = 2000,
    n_steps: int = 100,
    return_traj: bool = False,
    device: torch.device = device,
    rescale: float = RESCALE,
) -> Union[torch.Tensor, Tuple[torch.Tensor, torch.Tensor]]:
    """
    Euler integration of the induced velocity field b(t, x) = phi(t, t, x)
    from t=0 to t=1. Base samples match training: N(0, rescale^2 I).
    """
    model.eval()
    x = rescale * torch.randn(n, 2, device=device)
    dt = 1.0 / n_steps
    ts = torch.linspace(0.0, 1.0 - dt, n_steps, device=device)

    traj = [x.cpu()] if return_traj else None

    for t_val in ts:
        v = model.calc_b(t_val, x)   # b(t,x) = phi(t,t,x)
        x = x + dt * v
        if return_traj:
            traj.append(x.cpu())

    if return_traj:
        return x.cpu(), torch.stack(traj)   # (n_steps+1, n, 2)
    return x.cpu()


samples, trajectory = euler_sample(model, n=2000, n_steps=100, return_traj=True)
print(f"Generated samples: {samples.shape}  |  Trajectory: {trajectory.shape}")
print(f"sample std: {samples.std():.4f}  (target/rescale ~ {RESCALE:.4f})")

# ── Generated samples vs target ───────────────────────────────────────────────
from common.datasets import sample_two_moons

n_vis = 2000
x0_vis = trajectory[0].numpy()          # source noise from this euler run
x1_ref = sample_two_moons(n_vis)        # training target distribution

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].scatter(x0_vis[:, 0], x0_vis[:, 1], s=3, alpha=0.4, c="steelblue")
axes[0].set_title("Source $p_0$ (Gaussian)")

axes[1].scatter(samples[:, 0], samples[:, 1], s=3, alpha=0.4, c="mediumpurple")
axes[1].set_title("Generated $p_1^{\\theta}$ (Euler)")

axes[2].scatter(x1_ref[:, 0], x1_ref[:, 1], s=3, alpha=0.4, c="tomato")
axes[2].set_title("Target $p_1$ (two-moons)")

lim = 2.0
for ax in axes:
    ax.set_aspect("equal")
    ax.set_xlim(-lim, lim)
    ax.set_ylim(-lim, lim)

plt.tight_layout()
plt.show()

In [ ]:
# ── Trajectory plot: show a subset of particles moving from 0→1 ───────────────
TARGET_NAME = "two-moons"
n_show = 300
idx = np.random.choice(2000, n_show, replace=False)
traj_sub = trajectory[:, idx, :].numpy()   # (T, n_show, 2)
T = traj_sub.shape[0]

fig, ax = plt.subplots(figsize=(6, 6))
cmap = plt.get_cmap("cool")

for i in range(n_show):
    color = cmap(i / n_show)
    ax.plot(traj_sub[:, i, 0], traj_sub[:, i, 1],
            lw=0.5, alpha=0.4, color=color)

# start & end points
ax.scatter(*traj_sub[0].T, s=10, c="steelblue", zorder=5, label="$t=0$")
ax.scatter(*traj_sub[-1].T, s=10, c="tomato", zorder=5, label="$t=1$")
ax.set_aspect("equal")
ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim)
ax.set_title(f"ODE Trajectories (Gaussian → {TARGET_NAME})")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── Snapshot plots: distribution at fixed time steps ──────────────────────────
# Uses the full trajectory tensor (T+1, n, 2) from euler_sample above.
SNAP_TIMES = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]

T_steps = trajectory.shape[0] - 1          # number of Euler steps (= n_steps)
snap_indices = [round(t * T_steps) for t in SNAP_TIMES]

ncols = len(SNAP_TIMES)
fig, axes = plt.subplots(1, ncols, figsize=(3.2 * ncols, 3.4),
                         gridspec_kw={"wspace": 0.08})

for ax, step_idx, t_val in zip(axes, snap_indices, SNAP_TIMES):
    pts = trajectory[step_idx].numpy()      # (n, 2)
    ax.scatter(pts[:, 0], pts[:, 1], s=3, alpha=0.45,
               c=[t_val] * len(pts), cmap="cool", vmin=0, vmax=1)
    ax.set_title(f"$t = {t_val:.1f}$", fontsize=10)
    ax.set_aspect("equal")
    ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim)
    ax.set_xticks([]); ax.set_yticks([])

axes[0].set_ylabel("$x_2$")
fig.suptitle(f"Particle Distribution  (Gaussian → {TARGET_NAME})", fontsize=12, y=1.02)

# shared colorbar indicating time
sm = plt.cm.ScalarMappable(cmap="cool", norm=plt.Normalize(0, 1))
sm.set_array([])
fig.colorbar(sm, ax=axes, shrink=0.85, label="$t$", pad=0.02)
plt.show()

In [ ]:
from typing import Tuple

@torch.no_grad()
def eval_vf_on_grid(
    model,
    t_val: float,
    grid_size: int = 25,
    extent: float = 1.5,
    device: torch.device = device,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """Returns (X, Y, U, V) meshgrid arrays for a quiver plot of b(t, x) = phi(t, t, x)."""
    model.eval()
    xs = torch.linspace(-extent, extent, grid_size, device=device)
    ys = torch.linspace(-extent, extent, grid_size, device=device)
    XX, YY = torch.meshgrid(xs, ys, indexing="xy")
    grid = torch.stack([XX.flatten(), YY.flatten()], dim=1)
    uv = model.calc_b(torch.tensor(t_val, device=device), grid).cpu().numpy()
    X = XX.cpu().numpy()
    Y = YY.cpu().numpy()
    U = uv[:, 0].reshape(grid_size, grid_size)
    V = uv[:, 1].reshape(grid_size, grid_size)
    return X, Y, U, V


t_slices = [0.0, 0.25, 0.5, 0.75, 1.0]

fig, axes = plt.subplots(1, len(t_slices), figsize=(16, 3.5))

for ax, t_val in zip(axes, t_slices):
    X, Y, U, V = eval_vf_on_grid(model, t_val, grid_size=20, extent=lim)
    mag = np.sqrt(U**2 + V**2)
    ax.quiver(X, Y, U / (mag + 1e-8), V / (mag + 1e-8),
              mag, cmap="plasma", scale=25, width=0.004)
    ax.set_title(f"$b(x,\\, t={t_val:.2f})$", fontsize=11)
    ax.set_aspect("equal")
    ax.set_xlim(-lim, lim)
    ax.set_ylim(-lim, lim)
    ax.set_xticks([])
    ax.set_yticks([])

plt.suptitle("Induced velocity field $b(x, t)=\\phi(t,t,x)$ at fixed $t$", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── Reward: large on the upper moon, small elsewhere ─────────────────────────
# Uses an RBF envelope over reference points on sklearn's upper moon (label 0).
# Differentiable w.r.t. x → can be passed directly to the MPC terminal reward r(x).

import sys
sys.path.insert(0, "/home/shervink/flow_matching/flow-maps/py")
from common.datasets import sample_two_moons

REWARD_SIGMA = 0.35   # width of the bump (in data coordinates after centering/scaling)
N_REF        = 400    # reference points subsampled from the upper moon

# Build reference cloud using sample_two_moons (handles all normalization).
_all, _labels = sample_two_moons(50_000, return_labels=True)
upper_mask = _labels == 0
upper_pts  = torch.tensor(_all[upper_mask], dtype=torch.float32)
perm = torch.randperm(upper_pts.shape[0])[:N_REF]
UPPER_MOON_REF = upper_pts[perm]       # (N_REF, 2) — kept on CPU


def upper_moon_reward(x: torch.Tensor) -> torch.Tensor:
    """
    Terminal reward r(x): peaks at 1 on the upper moon, decays to ~0 elsewhere.

    r(x) = exp( - min_{p in upper_moon_ref} ||x - p||^2 / (2 σ^2) )

    Args:
        x: (B, 2) positions
    Returns:
        (B,) reward values in [0, 1]
    """
    ref = UPPER_MOON_REF.to(x.device, x.dtype)
    d2 = ((x[:, None, :] - ref[None, :, :]) ** 2).sum(dim=-1)   # (B, N_REF)
    return torch.exp(-d2.min(dim=1).values / (2.0 * REWARD_SIGMA ** 2))


# Alias for MPC usage:  receding_horizon_flow_control(..., r=upper_moon_reward, ...)
r = upper_moon_reward

# Terminal reward: prefer samples with large second coordinate.
# def r(x):
#     return 2.0 * x[:, 1]

# ── Contour visualisation ─────────────────────────────────────────────────────
EXTENT = 1.5
GRID   = 200
xs = np.linspace(-EXTENT, EXTENT, GRID)
ys = np.linspace(-EXTENT, EXTENT, GRID)
XX, YY = np.meshgrid(xs, ys, indexing="xy")
grid = torch.tensor(np.stack([XX.ravel(), YY.ravel()], axis=1), dtype=torch.float32)
R_grid = upper_moon_reward(grid).numpy().reshape(GRID, GRID)

# Full two-moons scatter for context.
_X_vis, _y_vis = sample_two_moons(3000, return_labels=True)
upper_vis = _y_vis == 0
lower_vis = _y_vis == 1

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5),
                         gridspec_kw={"wspace": 0.25},
                         subplot_kw={})
fig.delaxes(axes[1])
ax3 = fig.add_subplot(1, 2, 2, projection="3d")

# Left: filled contours + moon scatter
ax = axes[0]
cf = ax.contourf(XX, YY, R_grid, levels=30, cmap="YlOrRd", alpha=0.85)
fig.colorbar(cf, ax=ax, shrink=0.85, label="$r(x)$")
ax.contour(XX, YY, R_grid, levels=[0.1, 0.3, 0.5, 0.7, 0.9],
           colors="k", linewidths=0.6, alpha=0.5)
ax.scatter(_X_vis[lower_vis, 0], _X_vis[lower_vis, 1],
           s=4, alpha=0.35, c="steelblue", label="lower moon", zorder=3)
ax.scatter(_X_vis[upper_vis, 0], _X_vis[upper_vis, 1],
           s=4, alpha=0.55, c="tomato", label="upper moon (reward peak)", zorder=4)
ax.set_xlim(-EXTENT, EXTENT); ax.set_ylim(-EXTENT, EXTENT)
ax.set_aspect("equal")
ax.set_title(f"Reward contours  ($\\sigma={REWARD_SIGMA}$)")
ax.legend(loc="upper right", fontsize=8, markerscale=2)

# Right: 3-D surface
ax3.plot_surface(XX, YY, R_grid, cmap="YlOrRd", alpha=0.9,
                 linewidth=0, antialiased=True)
ax3.set_xlabel("$x_1$"); ax3.set_ylabel("$x_2$"); ax3.set_zlabel("$r(x)$")
ax3.set_title("Reward surface")
ax3.view_init(elev=35, azim=-60)

plt.suptitle("Upper-Moon Reward  (large on top arc, small elsewhere)", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

print(f"reward at upper-moon mean : {upper_moon_reward(upper_pts.mean(0, keepdim=True)).item():.4f}")
print(f"reward at origin          : {upper_moon_reward(torch.zeros(1, 2)).item():.4f}")

In [ ]:
"""
Receding-Horizon Path-Space Control for Flow Matching
=====================================================

Implements the softened path-space MPC variant with an INEQUALITY constraint
h(x) <= 0 enforced via a Lagrangian dual term and projected dual ascent:

    min_{x_{1:H}}  sum_{l=0}^{H-1} (1/(2*lam*dt)) * ||x_{l+1} - x_l - dt*v(t_{j+l}, x_l)||^2
                   - r(x_H) + <mu, h(z)>
    s.t. x_0 = x_cur

where z is either x_H (constraint_on="horizon") or the flow-map prediction of
the true endpoint X(t_{j+H}, 1; x_H) (constraint_on="endpoint").

After each inner solve, mu is updated via projected dual ascent:
    mu <- max(0, mu + lr_dual * h(z))

mu is a per-sample tensor of shape (B, C), one multiplier per particle per
constraint dimension.

You must supply:
    v_theta(t, x)      -> velocity,          shape (B, D)
    r(x)               -> terminal reward,   shape (B,)   (differentiable)
    h(x)               -> constraint vector, shape (B, C) (optional; h(x) <= 0)
    endpoint_map(t, x) -> flow-map to t=1,   shape (B, D) (required when
                          constraint_on="endpoint")
"""

import torch
from typing import Callable, Optional


@torch.no_grad()
def _rollout_uncontrolled(
    v_theta: Callable[[torch.Tensor, torch.Tensor], torch.Tensor],
    x0: torch.Tensor,
    t0_idx: int,
    H: int,
    dt: float,
    t_grid: torch.Tensor,
) -> torch.Tensor:
    """Warm start: roll the uncontrolled flow forward H steps.

    Returns x_bar of shape (H, B, D) — the states x_1, ..., x_H
    (x_0 is the fixed current state and is not included).
    """
    xs = []
    x = x0
    for ell in range(H):
        t = t_grid[t0_idx + ell]
        x = x + dt * v_theta(t, x)
        xs.append(x)
    return torch.stack(xs, dim=0)


def _horizon_objective(
    x_traj: torch.Tensor,                  # (H, B, D)
    x_cur: torch.Tensor,                   # (B, D), fixed
    v_theta: Callable,
    r: Callable,
    h: Optional[Callable],
    t0_idx: int,
    dt: float,
    lam: float,
    mu: Optional[torch.Tensor],            # (B, C) Lagrange multipliers, or None
    t_grid: torch.Tensor,
    constraint_on: str,                    # "horizon" | "endpoint"
    endpoint_map: Optional[Callable],
    t_horizon: Optional[torch.Tensor],     # scalar tensor for the horizon end-time
) -> torch.Tensor:
    """Lagrangian path-space objective, summed over the batch (scalar)."""
    H = x_traj.shape[0]
    x_full = torch.cat([x_cur.unsqueeze(0), x_traj], dim=0)  # (H+1, B, D)

    # Control-energy term
    energy = x_full.new_zeros(x_full.shape[1])  # (B,)
    for ell in range(H):
        t = t_grid[t0_idx + ell]
        resid = x_full[ell + 1] - x_full[ell] - dt * v_theta(t, x_full[ell])
        energy = energy + resid.pow(2).sum(dim=-1) / (2.0 * lam * dt)

    x_H = x_full[-1]
    obj = energy - r(x_H)  # (B,)

    # Inequality constraint term: <mu, h(z)>
    if h is not None and mu is not None:
        if constraint_on == "endpoint" and endpoint_map is not None:
            z = endpoint_map(t_horizon.expand(x_H.shape[0]), x_H)
        else:
            z = x_H
        hx = h(z)                               # (B, C)
        if hx.dim() == 1:
            hx = hx.unsqueeze(-1)
        obj = obj + (mu.detach() * hx).sum(dim=-1)

    return obj.sum()


def solve_horizon_problem(
    x_cur: torch.Tensor,
    v_theta: Callable,
    r: Callable,
    h: Optional[Callable],
    t0_idx: int,
    H: int,
    dt: float,
    lam: float,
    mu: Optional[torch.Tensor],
    t_grid: torch.Tensor,
    constraint_on: str,
    endpoint_map: Optional[Callable],
    optimizer: str = "lbfgs",
    n_iters: int = 50,
    lr: float = 0.1,
):
    """Solve the finite-horizon path-space problem.

    Returns:
        x1_star: (B, D)  first step of the optimal trajectory
        xH_star: (B, D)  last  step of the optimal trajectory (for dual update)
    """
    x_init = _rollout_uncontrolled(v_theta, x_cur, t0_idx, H, dt, t_grid)
    x_traj = x_init.clone().requires_grad_(True)
    x_cur_fixed = x_cur.detach()
    t_horizon = t_grid[t0_idx + H]

    kw = dict(
        x_cur=x_cur_fixed, v_theta=v_theta, r=r, h=h,
        t0_idx=t0_idx, dt=dt, lam=lam, mu=mu, t_grid=t_grid,
        constraint_on=constraint_on, endpoint_map=endpoint_map,
        t_horizon=t_horizon,
    )

    if optimizer.lower() == "lbfgs":
        opt = torch.optim.LBFGS(
            [x_traj], lr=lr, max_iter=n_iters,
            tolerance_grad=1e-8, tolerance_change=1e-10,
            line_search_fn="strong_wolfe",
        )
        def closure():
            opt.zero_grad()
            loss = _horizon_objective(x_traj, **kw)
            loss.backward()
            return loss
        opt.step(closure)
    else:  # adam
        opt = torch.optim.Adam([x_traj], lr=lr)
        for _ in range(n_iters):
            opt.zero_grad()
            _horizon_objective(x_traj, **kw).backward()
            opt.step()

    return x_traj[0].detach(), x_traj[-1].detach()  # x_1^*, x_H^*


def receding_horizon_flow_control(
    v_theta: Callable[[torch.Tensor, torch.Tensor], torch.Tensor],
    r: Callable[[torch.Tensor], torch.Tensor],
    x0: torch.Tensor,
    N: int = 100,
    H: int = 10,
    lam: float = 0.1,
    mu_init: float = 0.0,
    h: Optional[Callable[[torch.Tensor], torch.Tensor]] = None,
    constraint_on: str = "horizon",
    endpoint_map: Optional[Callable] = None,
    lr_dual: float = 0.0,
    optimizer: str = "lbfgs",
    n_iters: int = 50,
    lr: float = 0.1,
    verbose: bool = False,
    device: Optional[torch.device] = None,
):
    """
    Receding-Horizon Path-Space Control with inequality constraint h(x) <= 0.

    Args:
        v_theta:       velocity field  v_theta(t, x) -> (B, D).
        r:             terminal reward  r(x) -> (B,).
        x0:            initial samples from p_0, shape (B, D).
        N:             number of time steps (dt = 1/N).
        H:             planning horizon (clipped near t=1).
        lam:           control weight lambda > 0.
        mu_init:       initial scalar value for all Lagrange multipliers.
        h:             inequality constraint  h(x) -> (B, C),  h(x) <= 0.
                       Pass None to run unconstrained.
        constraint_on: where to evaluate h for the Lagrangian and dual update:
                       "horizon"  — evaluate at x_H  (default).
                       "endpoint" — evaluate at the flow-map prediction
                                    X(t_{j+H}, 1; x_H) of the true endpoint.
        endpoint_map:  Callable(t_batch, x) -> (B, D).  Required when
                       constraint_on="endpoint".
        lr_dual:       projected dual-ascent step size (>= 0).
                       0 keeps mu frozen at mu_init (fixed Lagrangian penalty).
        optimizer:     "lbfgs" or "adam" for the inner solve.
        n_iters:       inner-solver iterations per MPC step.
        lr:            inner-solver learning rate.
        verbose:       print per-step diagnostics.
        device:        torch device (defaults to x0.device).

    Returns:
        x_N:      final controlled samples,       shape (B, D).
        traj:     full controlled trajectory,     shape (N+1, B, D).
        controls: applied residual controls u_j*, shape (N, B, D).
        mu_final: final per-sample multipliers,   shape (B, C), or None.
    """
    if device is None:
        device = x0.device
    x0 = x0.to(device)
    dt = 1.0 / N
    t_grid = torch.linspace(0.0, 1.0, N + 1, device=device, dtype=x0.dtype)

    x = x0.detach().clone()
    traj = [x.clone()]
    controls = []

    # Initialise per-sample Lagrange multipliers mu_t: (B, C)
    mu_t = None
    if h is not None:
        with torch.no_grad():
            h0 = h(x0)
            C = h0.shape[-1] if h0.dim() > 1 else 1
        mu_t = torch.full(
            (x0.shape[0], C), float(mu_init), dtype=x0.dtype, device=device
        )

    for j in range(N):
        H_j = min(H, N - j)

        x1_star, xH_star = solve_horizon_problem(
            x, v_theta, r, h, t0_idx=j, H=H_j, dt=dt,
            lam=lam, mu=mu_t, t_grid=t_grid,
            constraint_on=constraint_on, endpoint_map=endpoint_map,
            optimizer=optimizer, n_iters=n_iters, lr=lr,
        )

        # ── Projected dual ascent ─────────────────────────────────────────────
        if h is not None and lr_dual > 0.0:
            with torch.no_grad():
                if constraint_on == "endpoint" and endpoint_map is not None:
                    t_h = t_grid[j + H_j]
                    z = endpoint_map(t_h.expand(x.shape[0]), xH_star)
                else:
                    z = xH_star
                hx = h(z)
                if hx.dim() == 1:
                    hx = hx.unsqueeze(-1)
                mu_t = (mu_t + lr_dual * hx).clamp_min(0.0)

        # ── Apply first MPC step ──────────────────────────────────────────────
        with torch.no_grad():
            v_j = v_theta(t_grid[j], x)
            u_star = (x1_star - x) / dt - v_j
            x = x + dt * (v_j + u_star)

        traj.append(x.clone())
        controls.append(u_star.clone())

        if verbose:
            mu_mean = mu_t.mean().item() if mu_t is not None else 0.0
            print(
                f"step {j+1:4d}/{N}  "
                f"|u*| mean = {u_star.norm(dim=-1).mean():.4e}  "
                f"mu mean = {mu_mean:.4e}"
            )

    return x, torch.stack(traj, dim=0), torch.stack(controls, dim=0), mu_t


# ─────────────────────────────────────────────────────────────────────────────
# Demo
# ─────────────────────────────────────────────────────────────────────────────

torch.manual_seed(0)
B, D = 256, 2
N_demo = 10

v_theta    = model.calc_b
mpc_device = next(model.parameters()).device

# Inequality constraint: x_0 <= 3  (h(x) = x[:,0:1] - 3 <= 0)
def h(x):
    return x[:, 0:1] - (-0.6)

# endpoint_map: model.forward(s, t, x) = X(s,t,x); use t=1 to predict x_N.
def endpoint_map(t_batch, x):
    ones = torch.ones_like(t_batch)
    return model(t_batch, ones, x)

x0 = torch.randn(B, D, device=mpc_device)

# Uncontrolled rollout for comparison.
dt_demo     = 1.0 / N_demo
t_grid_demo = torch.linspace(0.0, 1.0, N_demo + 1, device=mpc_device, dtype=x0.dtype)
with torch.no_grad():
    x_unc = x0.clone()
    traj_unc_list = [x_unc.clone()]
    for j in range(N_demo):
        x_unc = x_unc + dt_demo * v_theta(t_grid_demo[j], x_unc)
        traj_unc_list.append(x_unc.clone())
traj_unc = torch.stack(traj_unc_list, dim=0)  # (N+1, B, D)

xN, traj, controls, mu_final = receding_horizon_flow_control(
    v_theta, r, x0,
    N=N_demo, H=4,
    lam=2.0,
    mu_init=1.0,
    h = h,
    constraint_on="horizon",   # switch to "endpoint" to use X(x_H)
    endpoint_map=endpoint_map,
    lr_dual=0.5,
    optimizer="lbfgs", n_iters=30, lr=0.5,
    verbose=True, device=mpc_device,
)

print("\nfinal mean (controlled):  ", xN.mean(dim=0).tolist())
print("final mean (uncontrolled):", x_unc.mean(dim=0).tolist())
if mu_final is not None:
    print(f"mu_final  mean={mu_final.mean().item():.4f}  max={mu_final.max().item():.4f}")

# ── MPC visualisation ────────────────────────────────────────────────────────
mpc_metrics = visualize_control_results(
    traj_ctrl=traj,
    traj_unc=traj_unc,
    x0=x0,
    xN_ctrl=xN,
    xN_unc=x_unc,
    controls=controls,
    r=r,
    bg_pts=sample_target(4000),
    method_name="MPC",
    controlled_label="Controlled (MPC)",
    lam=2.0,
    controls_are_velocity=True,
    n_particles=256,
    show_snaps=True,
    snap_times=[0.0, 0.25, 0.5, 0.75, 1.0],
    lim=3.5,
    device=mpc_device,
)


## Visualization Helpers

In [ ]:
from typing import Callable, Optional, Sequence, Tuple, Union
import torch

def visualize_control_results(
    *,
    traj_ctrl: Union[torch.Tensor, np.ndarray],
    traj_unc: Union[torch.Tensor, np.ndarray],
    x0: Union[torch.Tensor, np.ndarray],
    xN_ctrl: Union[torch.Tensor, np.ndarray],
    xN_unc: Union[torch.Tensor, np.ndarray],
    controls: Optional[Union[torch.Tensor, np.ndarray]] = None,
    r: Optional[Callable[[torch.Tensor], torch.Tensor]] = None,
    bg_pts: Optional[np.ndarray] = None,
    method_name: str = "MPC",
    controlled_label: Optional[str] = None,
    lam: float = 2.0,
    controls_are_velocity: bool = False,
    n_particles: int = 256,
    show_snaps: bool = True,
    snap_times: Sequence[float] = (0.0, 0.25, 0.5, 0.75, 1.0),
    lim: float = 3.5,
    device: Optional[torch.device] = None,
    rng_seed: int = 42,
) -> Tuple[float, float, Optional[float], Optional[float]]:
    """Visualize controlled vs uncontrolled trajectories and control effort.

    Returns:
        (avg_r_unc, avg_r_ctrl, ctrl_cost_raw, ctrl_cost_obj)
        The last two entries are None when ``controls`` is not provided.
    """
    def _to_np(x):
        if isinstance(x, torch.Tensor):
            return x.detach().cpu().numpy()
        return np.asarray(x)

    traj_ctrl_np = _to_np(traj_ctrl)
    traj_unc_np  = _to_np(traj_unc)
    x0_np        = _to_np(x0)
    xN_ctrl_np   = _to_np(xN_ctrl)
    xN_unc_np    = _to_np(xN_unc)

    if bg_pts is None:
        bg_pts = sample_target(4000)

    if controlled_label is None:
        controlled_label = f"Controlled ({method_name})"

    rng_viz = np.random.default_rng(rng_seed)
    idx_show = rng_viz.choice(
        traj_ctrl_np.shape[1],
        size=min(n_particles, traj_ctrl_np.shape[1]),
        replace=False,
    )

    avg_r_unc = avg_r_ctrl = None
    if r is not None:
        if device is None:
            device = torch.device("cpu")
        avg_r_unc = r(torch.tensor(xN_unc_np, device=device)).mean().item()
        avg_r_ctrl = r(torch.tensor(xN_ctrl_np, device=device)).mean().item()

    # ── Figure 1: trajectory spaghetti ───────────────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(13, 5.5), sharey=True,
                             gridspec_kw={"wspace": 0.05})

    for ax, traj_data, pts_end, label, color in [
        (axes[0], traj_unc_np,  xN_unc_np,  "Uncontrolled",        "steelblue"),
        (axes[1], traj_ctrl_np, xN_ctrl_np, controlled_label,      "tomato"),
    ]:
        ax.scatter(bg_pts[:, 0], bg_pts[:, 1], s=2, alpha=0.10, c="gray", zorder=0)
        for i in idx_show:
            ax.plot(traj_data[:, i, 0], traj_data[:, i, 1],
                    lw=0.45, alpha=0.30, color=color)
        ax.scatter(x0_np[idx_show, 0], x0_np[idx_show, 1],
                   s=14, c="slategray", zorder=5, label="$t=0$")
        ax.scatter(pts_end[idx_show, 0], pts_end[idx_show, 1],
                   s=14, c=color, zorder=6, label="$t=1$")
        ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim)
        ax.set_aspect("equal")
        ax.tick_params(labelsize=8)
        ax.set_title(label, fontsize=11)
        ax.legend(fontsize=8, loc="upper left")

    if avg_r_unc is not None:
        fig.suptitle(
            f"{method_name} Trajectories  |  avg r: uncontrolled = {avg_r_unc:.3f}  "
            f"controlled = {avg_r_ctrl:.3f}",
            fontsize=12,
        )
    else:
        fig.suptitle(f"{method_name} Trajectories", fontsize=12)
    plt.tight_layout()
    plt.show()

    # ── Figure 2: snapshot grid ───────────────────────────────────────────────
    if show_snaps:
        ncols = len(snap_times)

        fig, axes = plt.subplots(2, ncols, figsize=(3.0 * ncols, 6.2),
                                 gridspec_kw={"hspace": 0.08, "wspace": 0.06})

        for row, (row_label, color, traj_data) in enumerate([
            (controlled_label, "tomato",    traj_ctrl_np),
            ("Uncontrolled",   "steelblue", traj_unc_np),
        ]):
            T_steps = max(traj_data.shape[0] - 1, 1)
            snap_idx = [
                min(round(t * T_steps), traj_data.shape[0] - 1)
                for t in snap_times
            ]
            for col, (sidx, t_val) in enumerate(zip(snap_idx, snap_times)):
                ax = axes[row, col]
                ax.scatter(bg_pts[:, 0], bg_pts[:, 1], s=2, alpha=0.10, c="gray", zorder=0)
                ax.scatter(traj_data[sidx, :, 0], traj_data[sidx, :, 1],
                           s=5, alpha=0.55, color=color)
                if row == 0:
                    ax.set_title(f"$t = {t_val:.2f}$", fontsize=10)
                ax.set_aspect("equal")
                ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim)
                ax.tick_params(labelsize=7)
            axes[row, 0].set_ylabel(row_label, fontsize=10)

        fig.suptitle("Snapshot Comparison  (gray = target)", fontsize=12, y=1.01)
        plt.tight_layout()
        plt.show()

    # ── Figure 3: control displacement + integrated cost ─────────────────────
    ctrl_cost_raw = ctrl_cost_obj = None
    if controls is not None:
        controls_np = _to_np(controls)
        N_steps = controls_np.shape[0]
        dt = 1.0 / N_steps
        dx_np = (dt * controls_np) if controls_are_velocity else controls_np
        dx_mag = np.linalg.norm(dx_np, axis=-1)
        t_axis = np.linspace(0.0, 1.0 - dt, N_steps)

        dx_sq_sum = (dx_mag ** 2).sum(axis=0)
        ctrl_cost_raw = float(dx_sq_sum.mean())
        ctrl_cost_obj = float((dx_sq_sum / (2.0 * lam)).mean())

        print(f"[{method_name}]  avg  Σ‖Δx‖²         = {ctrl_cost_raw:.4f}   (raw integrated effort)")
        print(f"[{method_name}]  avg  Σ‖Δx‖²/(2λ)  = {ctrl_cost_obj:.4f}   (control cost in objective, λ={lam})")

        fig, axes = plt.subplots(1, 2, figsize=(13, 3.2), gridspec_kw={"wspace": 0.3})

        ax = axes[0]
        ax.fill_between(t_axis,
                        np.percentile(dx_mag, 10, axis=1),
                        np.percentile(dx_mag, 90, axis=1),
                        alpha=0.25, color="tomato", label="10–90th pct")
        ax.plot(t_axis, dx_mag.mean(axis=1), color="tomato", lw=2,
                label="mean $\\|\\Delta x\\|$")
        ax.set_xlabel("$t$")
        ax.set_ylabel("$\\|\\Delta x(t)\\|$")
        ax.set_title(f"{method_name}: control displacement magnitude over time")
        ax.legend(fontsize=9)

        ax = axes[1]
        ax.hist(dx_sq_sum, bins=40, color="tomato", alpha=0.7, edgecolor="white")
        ax.axvline(ctrl_cost_raw, color="black", lw=1.5, linestyle="--",
                   label=f"mean = {ctrl_cost_raw:.3f}")
        ax.set_xlabel("$\\sum_j \\|\\Delta x_j\\|^2$")
        ax.set_ylabel("count")
        ax.set_title(f"{method_name}: integrated control effort (per sample)")
        ax.legend(fontsize=9)
        
        plt.tight_layout()
        plt.show()

    return avg_r_unc, avg_r_ctrl, ctrl_cost_raw

In [ ]:
"""
FMRG-J  (Flow Map Reward Guidance — Jacobian)
==============================================

At each step k we:
  1. Advance via the flow map:  x  ←  X_{t_k, t_{k+1}}(x)
  2. Apply n_opt gradient steps using the full-horizon Jacobian signal:
         u  =  ∇_x [ r( X_{t_{k+1},1}(x) ) ]
              =  J_X(x)^T  ∇r( X_{t_{k+1},1}(x) )
         x  ←  x  +  (Δt / n_opt) * λ * u

The Jacobian-transpose product is computed for free by backprop through
the single-step flow map  model(s, 1.0, x)  (which is X_{s,1} in closed form).

Signature of r:  r(x: Tensor[B, D]) -> Tensor[B]
Signature of lam: float  or  Callable(t, x) -> Tensor[B] or scalar
"""

from typing import Union, Callable, Optional


def fmrg_j(
    model,
    r: Callable,
    x0: torch.Tensor,
    *,
    N: int = 20,
    n_opt: int = 1,
    lam: Union[float, Callable] = 1.0,
    return_traj: bool = False,
    return_controls: bool = False,
    device: Optional[torch.device] = None,
):
    """
    FMRG-J: guided sampling via the Jacobian of the flow map.

    Args:
        model:      FlowMapMLP instance with forward(s, t, x) -> X_{s,t}(x).
        r:          differentiable reward  r(x) -> (B,).
        x0:         initial noise, shape (B, D), drawn from the base distribution.
        N:          number of time steps (dt = 1/N).
        n_opt:      inner gradient steps per outer time step.
        lam:        guidance strength: float, or callable lam(t, x) -> scalar/Tensor.
        return_traj: if True also return the full trajectory (N+1, B, D).
        device:     target device (defaults to x0.device).

    Returns:
        x1:   final guided samples, shape (B, D).
        traj: (optional) Tensor of shape (N+1, B, D).
    """
    if device is None:
        device = x0.device
    x = x0.clone().to(device)
    dt = 1.0 / N
    ts = torch.linspace(0.0, 1.0, N + 1, device=device, dtype=x.dtype)
    ones = ts[-1:]  # reused as t=1 below

    traj     = [x.cpu()] if return_traj else None
    controls = []                              # total guidance displacement per outer step

    for k in range(N):
        s = ts[k]
        t = ts[k + 1]
        B = x.shape[0]

        # ── Step 1: flow map advance  x ← X_{t_k, t_{k+1}}(x) ────────────────
        with torch.no_grad():
            x = model(s.expand(B), t.expand(B), x)

        # ── Step 2: n_opt Jacobian guidance steps ──────────────────────────────
        x_before = x.detach().clone()
        for _ in range(n_opt):
            x_in = x.detach().requires_grad_(True)

            # Full-horizon map X_{t_{k+1}, 1}(x) — single model call
            X1 = model(t.expand(B), ones.expand(B), x_in)   # (B, D)

            # u = J_X^T ∇r  via backprop through X1
            obj = r(X1).sum()
            u = torch.autograd.grad(obj, x_in)[0]            # (B, D)

            with torch.no_grad():
                strength = lam(t, x) if callable(lam) else lam
                x = x.detach() + (dt / n_opt) * strength * u

        # Record total displacement from guidance at this outer step
        controls.append((x.detach() - x_before).cpu())

        if return_traj:
            traj.append(x.cpu())

    result = x.detach()
    ctrl_tensor = torch.stack(controls, dim=0)  # (N, B, D)
    if return_traj and return_controls:
        return result, torch.stack(traj, dim=0), ctrl_tensor
    if return_traj:
        return result, torch.stack(traj, dim=0)
    if return_controls:
        return result, ctrl_tensor
    return result


# ── Demo ──────────────────────────────────────────────────────────────────────
B_demo = 512
x0_demo = RESCALE * torch.randn(B_demo, 2, device=device)

# Uncontrolled baseline (Euler on induced velocity, with trajectory)
@torch.no_grad()
def euler_baseline(x0, N=20, return_traj=False):
    x, dt = x0.clone(), 1.0 / N
    ts = torch.linspace(0.0, 1.0, N + 1, device=x.device)
    traj = [x.cpu()] if return_traj else None
    for k in range(N):
        x = x + dt * model.calc_b(ts[k], x)
        if return_traj:
            traj.append(x.cpu())
    if return_traj:
        return x.cpu(), torch.stack(traj, dim=0)
    return x.cpu()

xN_unc, traj_unc = euler_baseline(x0_demo, N=20, return_traj=True)

# FMRG-J controlled
xN_ctrl, traj_ctrl, controls_fmrg = fmrg_j(
    model, r, x0_demo,
    N=20, n_opt=3, lam=5.0,
    return_traj=True, return_controls=True, device=device,
)
xN_ctrl = xN_ctrl.cpu()

# Target reference
x1_ref, _ = sample_two_moons(B_demo, return_labels=True)


# ── FMRG-J visualisation ───────────────────────────────────────────────────
bg_pts_fmrg, _ = sample_two_moons(4000, return_labels=True)

fmrg_metrics = visualize_control_results(
    traj_ctrl=traj_ctrl,
    traj_unc=traj_unc,
    x0=x0_demo,
    xN_ctrl=xN_ctrl,
    xN_unc=xN_unc,
    controls=controls_fmrg,
    r=r,
    bg_pts=bg_pts_fmrg,
    method_name="FMRG-J",
    controlled_label="FMRG-J (controlled)",
    lam=2.0,
    controls_are_velocity=False,
    n_particles=128,
    show_snaps=True,
    snap_times=[0.0, 0.25, 0.5, 0.75, 1.0],
    lim=lim,
    device=device,
)

In [ ]:
torch.manual_seed(0)
B, D = 256, 2
N_demo = 10

# v_theta expects signature (t, x) -> (B, D); VectorField.forward is (x, t).
v_theta = model.calc_b   # b(t, x) = phi(t, t, x) — induced velocity field
mpc_device = next(model.parameters()).device

x0 = torch.randn(B, D, device=mpc_device)

# Uncontrolled rollout for comparison (same initial batch, no MPC).
dt_demo = 1.0 / N_demo
t_grid_demo = torch.linspace(0.0, 1.0, N_demo + 1, device=mpc_device, dtype=x0.dtype)
with torch.no_grad():
    x_unc = x0.clone()
    traj_unc_list = [x_unc.clone()]
    for j in range(N_demo):
        x_unc = x_unc + dt_demo * v_theta(t_grid_demo[j], x_unc)
        traj_unc_list.append(x_unc.clone())
traj_unc = torch.stack(traj_unc_list, dim=0)  # (N+1, B, D)

lambs = [0.01, 0.1, 0.25, 0.5, 0.75, 1.0, 1.25, 1.5, 1.75, 2.0, 3.0, 4.0, 5.0, 10.0]
rewards_mpc = np.zeros(len(lambs))
control_costs_mpc = np.zeros(len(lambs))
rewards_fmrg = np.zeros(len(lambs))
control_costs_fmrg = np.zeros(len(lambs))


for i, lamb in enumerate(lambs):

    xN, traj, controls, _ = receding_horizon_flow_control(
        v_theta, r, x0,
        N=N_demo, H=4, lam=lamb,
        optimizer="lbfgs", n_iters=30, lr=0.5,
        verbose=True, device=mpc_device,
    )

    print("\nfinal mean (controlled):  ", xN.mean(dim=0).tolist())
    print("final mean (uncontrolled):", x_unc.mean(dim=0).tolist())

    # ── MPC visualisation ────────────────────────────────────────────────────────
    mpc_metrics = visualize_control_results(
        traj_ctrl=traj,
        traj_unc=traj_unc,
        x0=x0,
        xN_ctrl=xN,
        xN_unc=x_unc,
        controls=controls,
        r=r,
        bg_pts=sample_target(4000),
        method_name="MPC",
        controlled_label="Controlled (MPC)",
        lam=2.0,
        controls_are_velocity=True,
        n_particles=256,
        show_snaps=True,
        snap_times=[0.0, 0.25, 0.5, 0.75, 1.0],
        lim=3.5,
        device=mpc_device,
    )
    
    rewards_mpc[i] = mpc_metrics[1]
    control_costs_mpc[i] = mpc_metrics[2]


    # FMRG-J controlled
    xN_ctrl, traj_ctrl, controls_fmrg = fmrg_j(
        model, r, x0,
        N=20, n_opt=3, lam=lamb,
        return_traj=True, return_controls=True, device=device,
    )
    xN_ctrl = xN_ctrl.cpu()

    # Target reference
    x1_ref, _ = sample_two_moons(B_demo, return_labels=True)


    # ── FMRG-J visualisation ───────────────────────────────────────────────────
    bg_pts_fmrg, _ = sample_two_moons(4000, return_labels=True)

    fmrg_metrics = visualize_control_results(
        traj_ctrl=traj_ctrl,
        traj_unc=traj_unc,
        x0=x0_demo,
        xN_ctrl=xN_ctrl,
        xN_unc=xN_unc,
        controls=controls_fmrg,
        r=r,
        bg_pts=bg_pts_fmrg,
        method_name="FMRG-J",
        controlled_label="FMRG-J (controlled)",
        lam=2.0,
        controls_are_velocity=False,
        n_particles=128,
        show_snaps=True,
        snap_times=[0.0, 0.25, 0.5, 0.75, 1.0],
        lim=lim,
        device=device,
    )

    rewards_fmrg[i] = fmrg_metrics[1]
    control_costs_fmrg[i] = fmrg_metrics[2]

plt.figure(figsize=(10, 5))
plt.scatter(control_costs_mpc, rewards_mpc, marker='o')
plt.scatter(control_costs_fmrg, rewards_fmrg, marker='x')
plt.legend(['MPC', 'FMRG-J'])
plt.xlabel('Control Cost')
plt.ylabel('Reward')

In [ ]:
plt.figure(figsize=(10, 5))
plt.scatter(control_costs_mpc, rewards_mpc, marker='o')
plt.scatter(control_costs_fmrg, rewards_fmrg, marker='x')
plt.xlim(-0.001, 0.125)

for x, y, lamb in zip(control_costs_mpc, rewards_mpc, lambs):
    plt.annotate(
        f"λ={lamb:g}",
        (x, y),
        textcoords="offset points",
        xytext=(4, 4),
        fontsize=7,
        color="C0",
    )

for x, y, lamb in zip(control_costs_fmrg, rewards_fmrg, lambs):
    plt.annotate(
        f"λ={lamb:g}",
        (x, y),
        textcoords="offset points",
        xytext=(4, -8),
        fontsize=7,
        color="C1",
    )


plt.legend(['MPC', 'FMRG-J'])
plt.xlabel('Control Cost')
plt.ylabel('Reward')